# Survival Analysis Script (Cox & Nelson–Aalen)

This script performs **survival analysis** using Cox proportional hazards models and Nelson–Aalen estimators.  
It should be executed **after the data cleaning process** to ensure that datasets are properly prepared.  

## What this script does
- **Cox Proportional Hazards Analysis:**  
  - Fits Cox models to evaluate the effect of features on survival time.  
  - Outputs hazard ratios and statistical significance.  

- **Nelson–Aalen Estimation:**  
  - Computes and plots Nelson–Aalen cumulative hazard curves. 


# Required Functions

In [ ]:
from functions_cox import *

# Prepare The Data

In [ ]:
cohort_years = pd.read_csv(
    r"../Data Cleaning/results/cohort_examination_years.csv",
    index_col="eid",
    parse_dates=["examination year"],
)
ad_years = pd.read_csv(r"ad_years.csv", index_col="eid", usecols=["eid", "ad_after0"])
ad_years.rename(columns={"ad_after0": "ad_after"}, inplace=True)
d_years = pd.read_csv(r"d_years.csv", index_col="eid", usecols=["eid", "ad_after"])
d_years.rename(columns={"ad_after": "d_after"}, inplace=True)
death_info = pd.read_csv(r"./death_record_table.csv", parse_dates=["Date of death"])
death_info.drop("dnx_death_id", axis=1, inplace=True)
death_info.rename(columns={"Participant ID": "eid"}, inplace=True)


ldf_ad_mm_knn = []
ldf_d_mm_knn = []
ldf_ad_mm_missing = []
ldf_d_mm_missing = []


MULTILEVEL_FEATURES = ["Educational status", "Alcohol Consumption", "Smoking Status"]

LABEL_MAPS = {
    "Educational status": {
        0.0: "Higher",
        1.0: "Upper secondary",
        2.0: "Lower secondary",
        3.0: "Vocational",
        4.0: "Other",
    },
    "Alcohol Consumption": {
        0.0: "Daily",
        1.0: "3 or 4 /week",
        2.0: "1 or 2/week",
        3.0: "1-3/month",
        4.0: "Special occations",
        5.0: "Never",
    },
    "Smoking Status": {
        0.0: "Never",
        1.0: "Previous",
        2.0: "Current",
    },
}


def one_hot(df):
    df = df.copy()
    for col, mapping in LABEL_MAPS.items():
        df[col] = pd.Categorical(
            df[col].map(mapping),
            categories=list(mapping.values()),
            ordered=False,
        )
    return pd.get_dummies(df, columns=MULTILEVEL_FEATURES, drop_first=True, dtype=float)


for i in range(1, 11):
    knn_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_age_matched_knn.csv".format(i),
        index_col="eid",
    )
    missing_mm = pd.read_csv(
        r"../Data Cleaning/results_{}/missing_matched_age_matched_missing.csv".format(
            i
        ),
        index_col="eid",
    )

    ad_mm_knn = one_hot(process_ad(knn_mm, death_info, ad_years, cohort_years))
    d_mm_knn = one_hot(process_d(knn_mm, death_info, d_years, cohort_years))
    ad_mm_missing = one_hot(process_ad(missing_mm, death_info, ad_years, cohort_years))
    d_mm_missing = one_hot(process_d(missing_mm, death_info, d_years, cohort_years))

    ldf_ad_mm_knn.append(ad_mm_knn)
    ldf_d_mm_knn.append(d_mm_knn)
    ldf_ad_mm_missing.append(ad_mm_missing)
    ldf_d_mm_missing.append(d_mm_missing)


dummy_cols = [
    c
    for c in ldf_ad_mm_knn[0].columns
    if any(c.startswith(m + "_") for m in MULTILEVEL_FEATURES)
]

CAT_FEATURES = ["Sex", "Diabetes", "Antihypertensive usage"] + dummy_cols

NUM_FEATURES = ldf_ad_mm_knn[0].columns.difference(CAT_FEATURES + ["Status", "eid"])

# KaplanMeier

Since we don’t have the true survival curve of the population, thus we will estimate the survival curve from the data using Non-parametric Kaplan-Meier estimator

$$\hat{S}(t) = \prod_{t_i \le t}\left(1 - \frac{d_i}{n_i}\right)$$



In [ ]:
kmf = KaplanMeierFitter()

kmf.fit(d_mm_knn["d_after"], d_mm_knn["Status"])
kmf.survival_function_.plot()
plt.title("Survival function of Dementia")

y axis represents the probability a cohort doesn't have dementia after t years, where t years is on the x-axis.

In [ ]:
kmf.plot_survival_function()
plt.title("Survival function of AD")

In [ ]:
# get the median survival time
median_ci = median_survival_times(kmf.confidence_interval_)

T = d_mm_knn["d_after"]
E = d_mm_knn["Status"]

rnfl_1 = d_mm_knn["mRNFL thickness"] < d_mm_knn["mRNFL thickness"].median()

fig, ax = plt.subplots(1, 1)

kmf.fit(T[rnfl_1], E[rnfl_1], label="mRNFL thickness < median")
kmf.survival_function_.plot(ax=ax)

kmf.fit(T[~rnfl_1], E[~rnfl_1], label="mRNFL thickness >= median")
kmf.survival_function_.plot(ax=ax)

# Nelson Aalen

If we are curious about the hazard function h(t) of a population, we unfortunately cannot transform the Kaplan Meier estimate — statistics doesn’t work quite that well. Fortunately, there is a proper non-parametric estimator of the cumulative hazard function:

$$\hat{H}(t) = \sum_{t_i \le t}\frac{d_i}{n_i}$$

In [ ]:
naf = NelsonAalenFitter()

T = d_mm_knn["d_after"]
E = d_mm_knn["Status"]

naf.fit(T, event_observed=E)

naf.plot_cumulative_hazard()

In [ ]:
T = d_mm_knn["d_after"]
E = d_mm_knn["Status"]

rnfl_1 = d_mm_knn["mRNFL thickness"] < d_mm_knn["mRNFL thickness"].median()

fig, ax = plt.subplots(1, 1)

naf.fit(T[rnfl_1], event_observed=E[rnfl_1], label="mRNFL thickness < median")
ax = naf.cumulative_hazard_.plot(ax=ax)

naf.fit(T[~rnfl_1], event_observed=E[~rnfl_1], label="mRNFL thickness >= median")
naf.cumulative_hazard_.plot(ax=ax)

plt.title("Cumulative hazard of AD by mRNFL thickness")
plt.ylabel("Cumulative hazard")
plt.xlabel("Time (years)")
plt.legend()

# Weibull model

Another very popular model for survival data is the Weibull model. In contrast the the Nelson-Aalen estimator, this model is a parametric model, meaning it has a functional form with parameters that we are fitting the data to. (The Nelson-Aalen estimator has no parameters to fit to). The survival function looks like:


$$S(t) = \exp\!\left(-\left(t/\lambda\right)^{\rho}\right)$$

In [ ]:
import os

from lifelines import WeibullFitter

fig, ax = plt.subplots()

T = d_mm_knn["d_after"] + 5
E = d_mm_knn["Status"]
rnfl_1 = d_mm_knn["mRNFL thickness"] < d_mm_knn["mRNFL thickness"].median()

wbf1 = WeibullFitter()
wbf2 = WeibullFitter()

wbf1.fit(T[rnfl_1], event_observed=E[rnfl_1], label="thin")
wbf1.plot_cumulative_hazard(ax=ax, color="red")
wbf1.fit(T[~rnfl_1], event_observed=E[~rnfl_1], label="tick")
wbf1.plot_cumulative_hazard(ax=ax, color="blue")


# COX

The idea behind Cox’s proportional hazard model is that the log-hazard of an individual is a linear function of their covariates and a population-level baseline hazard that changes over time. Mathematically:

$$h(t \mid x) = h_0(t)\,\exp(\beta_1 x_1 + \cdots + \beta_p x_p)$$

# Dementia

## missing matched

In [ ]:
ldf_d_mm_knn[0]

In [ ]:
cox_out_path = "./results_combined/final/cox/"
os.makedirs(cox_out_path, exist_ok=True)

agg_df_d = fit_cox_rubins(ldf_d_mm_knn, "d_after", "Status", CAT_FEATURES)
agg_df_d.to_csv(cox_out_path + "cox_summary_d_mm.csv")
agg_df_d

In [ ]:
plot_cox_forest(agg_df_d, save_path=cox_out_path + "cox_plot_dementia_mm.pdf")

In [ ]:
plot_all_features_on_single_figure(
    ldf_d_mm_knn,
    [
        "mGCIPL thickness",
        "mRNFL thickness",
        "Total macular volume",
        "Average RPE thickness",
        "Overall macular thickness",
    ],
    CAT_FEATURES,
    output_path=cox_out_path + "dementia_mm_",
    task="dementia",
)

# AD

# mm

In [ ]:
agg_df_ad = fit_cox_rubins(ldf_ad_mm_knn, "ad_after", "Status", CAT_FEATURES)
agg_df_ad.to_csv(cox_out_path + "cox_summary_ad_mm.csv")
agg_df_ad

In [ ]:
important_features = get_important_cox_features(agg_df_ad, p_threshold=0.05)
print("Important features with p-value < 0.05:")
print(important_features)

In [ ]:
plot_cox_forest(agg_df_ad, save_path=cox_out_path + "cox_plot_ad_mm.pdf")

In [ ]:
plot_all_features_on_single_figure(
    ldf_ad_mm_knn,
    [
        "mGCIPL thickness",
        "mRNFL thickness",
        "Total macular volume",
        "Average RPE thickness",
        "Overall macular thickness",
    ],
    CAT_FEATURES,
    output_path=cox_out_path + "ad_mm_",
    task="ad",
)